# 05 - Baselines

Computes refusal direction (Arditi et al.) baseline AUROC and vanilla Gemma evaluation for comparison.

In [ ]:
import os
import sys
REPO_DIR = '/kaggle/working/pc-sae'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from src.utils import setup_hf_auth
setup_hf_auth()

In [ ]:
!python scripts/05_run_baselines.py

In [ ]:
from src.utils import METRICS, load_json
baselines = load_json(METRICS / 'baselines.json')
summary = load_json(METRICS / 'evaluation_summary.json')
training = load_json(METRICS / 'pc_training.json')

print('=== COMPARISON ===\n')
print(f"Density AUROC:")
if baselines.get('refusal_direction'):
    print(f"  refusal direction baseline: {baselines['refusal_direction']['auroc']:.4f}")
print(f"  PC-SAE monitor:             {summary['density_auroc']:.4f}")
print(f"\nJailbreakBench ASR:")
if baselines.get('vanilla_jailbreakbench', {}).get('asr') is not None:
    print(f"  vanilla (no monitor): {baselines['vanilla_jailbreakbench']['asr']:.3f}")
print(f"  with PC-SAE monitor:  {summary['jailbreakbench_asr']:.3f}")
print(f"\nXSTest over-refusal:")
if baselines.get('vanilla_xstest', {}).get('over_refusal_rate') is not None:
    print(f"  vanilla:             {baselines['vanilla_xstest']['over_refusal_rate']:.3f}")
print(f"  with PC-SAE monitor: {summary['xstest_overrefusal']:.3f}")

In [ ]:
from src.utils import METRICS, load_json
import pandas as pd

db = load_json(METRICS / 'density_baselines.json')
rows = []
for method, v in db.items():
    if isinstance(v, dict) and 'auroc' in v:
        rows.append({'method': method, 'density_auroc': v['auroc']})
df = pd.DataFrame(rows).sort_values('density_auroc', ascending=False, na_position='last')
print('=== DENSITY AUROC: PC-SAE vs baselines (same split, no leakage) ===')
print('(logistic_probe_supervised sees unsafe in train: supervised ceiling, NOT anomaly detection)')
df.reset_index(drop=True)

In [ ]:
!cat outputs/logs/baselines.log 2>/dev/null | tail -40